In [10]:
import h5py
import numpy as np
import xarray as xr

# Process .mat file and convert to .nc 

file_path = "/glade/work/awells/air_quality/BMR/GBD_Region_Masks_0.10.mat"

with h5py.File(file_path, "r") as f:
    # Dereference mask objects
    mask_refs = f["sGBDRegions/Mask"][0]
    name_refs = f["sGBDRegions/name"][0]

    lat = f["gLAT"][0] 
    lon = f["gLON"][0]

    masks = []
    names = []
    regions = []

    for i, ref in enumerate(mask_refs):
        # Dereference mask
        mask = f[ref][()]
        masks.append(mask)

        # Country name
        name_ascii = f[name_refs[i]][()]
        name_str = "".join([chr(c[0]) for c in name_ascii])
        names.append(name_str)

# Confirm all masks have same shape before stacking
first_shape = masks[0].shape
if all(mask.shape == first_shape for mask in masks):
    # Stack into 3D array: (country, lat, lon)
    mask_array = np.stack(masks)
    data_array = xr.DataArray(
        mask_array,
        dims=["region", "lat", "lon"],
        coords={
            "region": names,
            "lat": lat,
            "lon": lon,
        },
        name="region_mask"
    )
else:
    raise ValueError("Masks have varying shapes – cannot stack into DataArray.")

data_array.to_netcdf("/glade/work/awells/air_quality/BMR/GBD_Region_Masks_0.10.nc")